## Imports

In [53]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict
from torch.utils.data import Dataset
import torch
from torch.utils.data import random_split, DataLoader
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from tqdm import tqdm
import matplotlib.pyplot as plt


## DATA STRUCTURES FOR RESNET DATASET


In [56]:
@dataclass
class CropSample:
    """
    Represents a single crop sample for ResNet training.
    
    Attributes:
        image_path: Path to the original source image
        bbox: Bounding box coordinates (x1, y1, x2, y2)
        category_id: Category ID (0-4) for first stage classification
        sign_class: Specific sign class (0-54) or -1 for background
        is_background: Flag indicating if this is a background/negative sample
    """
    image_path: Path
    bbox: Tuple[int, int, int, int]
    category_id: int
    sign_class: int  # -1 for background
    is_background: bool


@dataclass
class CategoryDataset:
    """
    Dataset for a single ResNet category.
    
    Attributes:
        category_id: Category ID (0-4)
        samples: List of crop samples belonging to this category
    """
    category_id: int
    samples: List[CropSample]


## IOU CALCULATION


In [59]:
def iou(box1: List[int], box2: List[int]) -> float:
    """
    Calculate Intersection over Union (IoU) between two bounding boxes.
    
    Args:
        box1: First bounding box [x1, y1, x2, y2]
        box2: Second bounding box [x1, y1, x2, y2]
    
    Returns:
        IoU value between 0 and 1
    """
    # Calculate intersection coordinates
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Calculate intersection area
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # Calculate union area
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0


def count_boxes(label_dir):
    label_dir = Path(label_dir)
    total = 0
    for label_file in label_dir.glob("*.txt"):
        with open(label_file, 'r') as f:
            total += len(f.readlines())
    return total


## COLLECT RANDOM BACKGROUND CROPS

In [62]:
def collect_random_background_crops(
    image_dir: str,
    ground_truths_map: Dict[str, List[dict]],
    num_per_image: int = 5,
    crop_size: tuple = (100, 100)
) -> List[CropSample]:
    """
    Collect random background crops from areas without signs.
    
    Args:
        image_dir: Directory containing images
        ground_truths_map: Mapping from image name to ground truth boxes
        num_per_image: Number of random crops per image
        crop_size: Size of random crops (width, height)
    
    Returns:
        List of background CropSample objects
    """
    random_bg_samples = []
    
    for img_path in Path(image_dir).glob("*.jpg"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
            
        h, w = img.shape[:2]
        
        # Get ground truths for this image
        gts = ground_truths_map.get(img_path.stem, [])
        
        for _ in range(num_per_image):
            # Generate random position
            x1 = random.randint(0, max(1, w - crop_size[0]))
            y1 = random.randint(0, max(1, h - crop_size[1]))
            x2 = x1 + crop_size[0]
            y2 = y1 + crop_size[1]
            random_bbox = [x1, y1, x2, y2]
            
            # Check if random crop overlaps with any ground truth sign
            is_overlap = False
            for gt in gts:
                if iou(random_bbox, gt["bbox"]) > 0.1:
                    is_overlap = True
                    break
            
            # If no overlap, add as background for all categories
            if not is_overlap:
                for cat_id in range(5):
                    random_bg_samples.append(
                        CropSample(
                            image_path=img_path,
                            bbox=tuple(random_bbox),
                            category_id=cat_id,
                            sign_class=-1,
                            is_background=True
                        )
                    )
    
    return random_bg_samples

## MAIN DATASET BUILDER


In [4]:
def build_resnet_dataset(
    image_dir: str,
    label_dir: str,
    yolo_model: YOLO,
    sign_to_category: Dict[int, int],  # sign_class -> category_id mapping
    iou_threshold: float = 0.5,
    background_ratio: float = 0.3  # Target background proportion in dataset
) -> Dict[int, CategoryDataset]:
    """
    Build dataset for 5 ResNet models by matching YOLO predictions with ground truth.
    
    This function:
    1. Runs YOLO inference on all images
    2. Matches predictions with ground truth using IoU
    3. Creates positive samples for correct predictions
    4. Creates background samples for false positives and missed signs
    5. Balances the dataset by limiting background samples
    
    Args:
        image_dir: Directory with training images
        label_dir: Directory with YOLO format labels
        yolo_model: Loaded YOLO model
        sign_to_category: Mapping from sign class (0-54) to category (0-4)
        iou_threshold: IoU threshold for matching predictions with ground truth
        background_ratio: Desired proportion of background samples in dataset
    
    Returns:
        Dictionary mapping category_id to CategoryDataset
    """
    
    # Initialize datasets for each category
    category_datasets = {
        cat_id: CategoryDataset(category_id=cat_id, samples=[])
        for cat_id in range(5)
    }
    
    # Track false positives for each category (to use as background)
    false_positives_by_category = defaultdict(list)
    
    # Build ground truth map for random background collection
    ground_truths_map = {}
    
    # Process each image
    for img_path in Path(image_dir).glob("*.jpg"):
        #print(f"Processing: {img_path.name}")
        
        # Load image
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Warning: Could not read {img_path}")
            continue
            
        h, w = img.shape[:2]
        
        # ============================================================
        # Step 1: Get YOLO predictions
        # ============================================================
        results = yolo_model(img, conf=0.5, verbose=False)
        predictions = []
        
        if results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                predictions.append({
                    "bbox": [x1, y1, x2, y2],
                    "confidence": float(box.conf[0]),
                    "yolo_category": int(box.cls[0])  # Category from YOLO
                })
        
        # ============================================================
        # Step 2: Read ground truth labels
        # ============================================================
        label_path = Path(label_dir) / (img_path.stem + ".txt")
        ground_truths = []
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    
                    
                    sign_class = int(float(parts[0]))  # 0-54
                    x_c, y_c, box_w, box_h = map(float, parts[1:5])
                    
                    # Convert normalized coordinates to pixel coordinates
                    x1 = int((x_c - box_w / 2) * w)
                    y1 = int((y_c - box_h / 2) * h)
                    x2 = int((x_c + box_w / 2) * w)
                    y2 = int((y_c + box_h / 2) * h)
                    
                    ground_truths.append({
                        "bbox": [x1, y1, x2, y2],
                        "sign_class": sign_class,
                        "category_id": sign_to_category[sign_class]
                    })
        
        # Store ground truths for random background collection
        ground_truths_map[img_path.stem] = ground_truths
        
        # ============================================================
        # Step 3: Match predictions with ground truth
        # ============================================================
        matched_gt_indices = set()
        
        for pred in predictions:
            pred_bbox = pred["bbox"]
            best_match_idx = None
            best_iou = 0
            
            for i, gt in enumerate(ground_truths):
                current_iou = iou(pred_bbox, gt["bbox"])
                if current_iou > iou_threshold and current_iou > best_iou:
                    best_iou = current_iou
                    best_match_idx = i
            
            if best_match_idx is not None:
                # TRUE POSITIVE: Add to dataset with correct sign class
                gt = ground_truths[best_match_idx]
                matched_gt_indices.add(best_match_idx)
                
                crop_sample = CropSample(
                    image_path=img_path,
                    bbox=tuple(pred_bbox),
                    category_id=gt["category_id"],
                    sign_class=gt["sign_class"],
                    is_background=False
                )
                category_datasets[gt["category_id"]].samples.append(crop_sample)
                
            else:
                # FALSE POSITIVE: YOLO detected something that isn't there
                # Add as background for the category YOLO predicted
                yolo_category = pred["yolo_category"]
                false_positives_by_category[yolo_category].append(
                    CropSample(
                        image_path=img_path,
                        bbox=tuple(pred_bbox),
                        category_id=yolo_category,
                        sign_class=-1,  # Background
                        is_background=True
                    )
                )
        
        # ============================================================
        # Step 4: Handle false negatives (signs that YOLO missed)
        # ============================================================
        for i, gt in enumerate(ground_truths):
            if i not in matched_gt_indices:
                # YOLO missed this sign - add as background for its category
                false_positives_by_category[gt["category_id"]].append(
                    CropSample(
                        image_path=img_path,
                        bbox=tuple(gt["bbox"]),
                        category_id=gt["category_id"],
                        sign_class=-1,
                        is_background=True
                    )
                )
    
    # ============================================================
    # Step 5: Add random background crops
    # ============================================================
    print("\nCollecting random background crops...")
    random_bg_samples = collect_random_background_crops(
        image_dir, ground_truths_map, num_per_image=5
    )
    
    for cat_id in range(5):
        category_samples = [s for s in random_bg_samples if s.category_id == cat_id]
        category_datasets[cat_id].samples.extend(category_samples)
    
    # ============================================================
    # Step 6: Add false positives to datasets
    # ============================================================
    for cat_id, fp_samples in false_positives_by_category.items():
        # Only add false positives that belong to this category
        category_datasets[cat_id].samples.extend(fp_samples)
    
    # ============================================================
    # Step 7: Balance the dataset (limit background samples)
    # ============================================================
    print("\nDataset statistics after building:")
    
    total_positive = 0
    total_background = 0
    
    for cat_id in range(5):
        dataset = category_datasets[cat_id]
        positive_samples = [s for s in dataset.samples if not s.is_background]
        background_samples = [s for s in dataset.samples if s.is_background]
        
        # Limit background to maintain background_ratio
        if len(positive_samples) > 0:
            max_bg = int(len(positive_samples) * background_ratio / (1 - background_ratio))
            if len(background_samples) > max_bg:
                background_samples = random.sample(background_samples, max_bg)
        
        dataset.samples = positive_samples + background_samples
        
        total_positive += len(positive_samples)
        total_background += len(background_samples)
        
        print(f"Category {cat_id}: {len(positive_samples)} positive, {len(background_samples)} background")
    
    # Print totals
    print("-" * 40)
    print(f"TOTAL:        {total_positive} positive, {total_background} background")
    print(f"GRAND TOTAL:  {total_positive + total_background} samples")
    print("=" * 40)
    
    return category_datasets

NameError: name 'YOLO' is not defined

## PYTORCH DATASET WRAPPER


In [68]:
class ResNetDataset(Dataset):
    """
    PyTorch Dataset for ResNet training.
    Loads cropped sign images from disk on-the-fly.
    """
    
    def __init__(self, samples, num_classes, input_size=224, augment=True):
        """
        Args:
            samples: List of CropSample objects
            num_classes: Total classes (signs + 1 background)
            input_size: Resize crops to this size (default: 224)
            augment: Apply data augmentation (default: True)
        """
        self.samples = samples
        self.num_classes = num_classes
        self.input_size = input_size
        self.augment = augment
        
        # Background is always the last class
        self.bg_class = num_classes - 1
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        # Get sample
        sample = self.samples[idx]
        
        # Load image
        img = cv2.imread(str(sample.image_path))
        
        # Extract crop using bounding box
        x1, y1, x2, y2 = sample.bbox
        crop = img[y1:y2, x1:x2]
        
        # Resize to ResNet input size
        crop = cv2.resize(crop, (self.input_size, self.input_size))
        
        # Convert BGR (OpenCV) to RGB (PyTorch)
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        
        # Apply augmentation if enabled
        if self.augment:
            crop = self._augment(crop)
        
        # Convert to tensor: (H, W, C) -> (C, H, W) and scale to [0, 1]
        crop = torch.from_numpy(crop).permute(2, 0, 1).float() / 255.0
        
        # Target class
        if sample.is_background:
            target = self.bg_class  # Background = last class
        else:
            target = sample.sign_class  # Regular sign (0, 1, 2, ...)
        
        return crop, target
    
    def _augment(self, img):
        """Simple augmentation: horizontal flip and rotation"""
        import random
        
        # Random horizontal flip (50% chance)
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
        
        # Random rotation (-10 to +10 degrees)
        if random.random() > 0.5:
            angle = random.uniform(-10, 10)
            h, w = img.shape[:2]
            center = (w // 2, h // 2)
            matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
            img = cv2.warpAffine(img, matrix, (w, h))
        
        return img


## Dataloader

In [71]:
def create_dataloader(category_dataset, batch_size=32, ):
    """
    Create train and validation dataloaders for one category.
    """
    
    samples = category_dataset.samples
    
    # Find number of classes
    max_class = -1
    for s in samples:
        if not s.is_background and s.sign_class > max_class:
            max_class = s.sign_class
    
    # Total classes = max_class + 1 (for 0-index) + 1 (for background)
    num_classes = max_class + 2
    
    print(f"Category {category_dataset.category_id}: {num_classes - 1} sign classes + 1 background")
    
    # Create dataset
    dataset = ResNetDataset(samples, num_classes=num_classes)
    
    # Split train/val
    train_size = int(train_split * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    
    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    return dataloader


## SIMPLE PREDICTION FUNCTION

In [74]:

# ============================================================
# MAPPING: 55 SIGN NAMES (global order 0-54)
# ============================================================

ALL_SIGNS = [
    'forb_ahead', 'forb_left', 'forb_overtake', 'forb_right', 'forb_speed_over_10',
    'forb_speed_over_100', 'forb_speed_over_130', 'forb_speed_over_20', 'forb_speed_over_30',
    'forb_speed_over_40', 'forb_speed_over_5', 'forb_speed_over_50', 'forb_speed_over_60',
    'forb_speed_over_70', 'forb_speed_over_80', 'forb_speed_over_90', 'forb_stopping',
    'forb_trucks', 'forb_u_turn', 'forb_weight_over_3.5t', 'forb_weight_over_7.5t',
    'info_bus_station', 'info_crosswalk', 'info_highway', 'info_one_way_traffic', 'info_parking',
    'info_taxi_parking', 'mand_bike_lane', 'mand_left', 'mand_left_right', 'mand_pass_left',
    'mand_pass_left_right', 'mand_pass_right', 'mand_right', 'mand_roundabout', 'mand_straigh_left',
    'mand_straight', 'mand_straight_right', 'prio_give_way', 'prio_priority_road', 'prio_stop',
    'warn_children', 'warn_construction', 'warn_crosswalk', 'warn_cyclists', 'warn_domestic_animals',
    'warn_other_dangers', 'warn_poor_road_surface', 'warn_roundabout', 'warn_slippery_road',
    'warn_speed_bumper', 'warn_traffic_light', 'warn_tram', 'warn_two_way_traffic', 'warn_wild_animals'
]

NAME_TO_ID = {name: idx for idx, name in enumerate(ALL_SIGNS)}


# ============================================================
# CATEGORY TO SIGNS MAPPING
# ============================================================

CATEGORY_SIGNS = {
    0: [
        'warn_children', 'warn_construction', 'warn_crosswalk', 'warn_cyclists',
        'warn_domestic_animals', 'warn_other_dangers', 'warn_poor_road_surface',
        'warn_roundabout', 'warn_slippery_road', 'warn_speed_bumper', 'warn_traffic_light',
        'warn_tram', 'warn_two_way_traffic', 'warn_wild_animals'
    ],
    1: [
        'forb_ahead', 'forb_left', 'forb_overtake', 'forb_right', 'forb_speed_over_10',
        'forb_speed_over_100', 'forb_speed_over_130', 'forb_speed_over_20', 'forb_speed_over_30',
        'forb_speed_over_40', 'forb_speed_over_5', 'forb_speed_over_50', 'forb_speed_over_60',
        'forb_speed_over_70', 'forb_speed_over_80', 'forb_speed_over_90', 'forb_stopping',
        'forb_trucks', 'forb_u_turn', 'forb_weight_over_3.5t', 'forb_weight_over_7.5t'
    ],
    2: [
        'info_bus_station', 'info_crosswalk', 'info_highway', 'info_one_way_traffic',
        'info_parking', 'info_taxi_parking'
    ],
    3: [
        'mand_bike_lane', 'mand_left', 'mand_left_right', 'mand_pass_left',
        'mand_pass_left_right', 'mand_pass_right', 'mand_right', 'mand_roundabout',
        'mand_straigh_left', 'mand_straight', 'mand_straight_right'
    ],
    4: [
        'prio_give_way', 'prio_priority_road', 'prio_stop'
    ]
}


# ============================================================
# IMAGENET NORMALIZATION (для CPU)
# ============================================================

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])


def normalize_image(img_tensor):
    """Normalize image tensor (C, H, W) using ImageNet stats"""
    return (img_tensor - IMAGENET_MEAN.view(3, 1, 1)) / IMAGENET_STD.view(3, 1, 1)


# ============================================================
# PREDICTION FUNCTION (CPU ONLY)
# ============================================================

def predict_signs(
    yolo: YOLO,
    resnets: Dict[int, torch.nn.Module],
    image: np.ndarray
) -> List[Dict]:
    """
    Two-stage detection on CPU.
    
    Args:
        yolo: YOLO model (automatically uses CPU if no CUDA)
        resnets: Dict of 5 ResNet models (should be on CPU)
        image: Input image (numpy array, BGR)
    
    Returns:
        List of detections with bbox, class_id, class_name, confidence
    """
    
    # Stage 1: YOLO (CPU)
    results = yolo(image)[0]
    
    if results.boxes is None:
        return []
    
    detections = []
    
    for box in results.boxes:
        # Get YOLO prediction
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        category = int(box.cls[0])
        
        # Extract crop
        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        
        # Get ResNet for this category
        resnet = resnets.get(category)
        if resnet is None:
            continue
        
        # Prepare crop
        crop = cv2.resize(crop, (224, 224))
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        
        # Convert to tensor
        crop_tensor = torch.from_numpy(crop).permute(2, 0, 1).float() / 255.0
        
        # Normalize
        crop_tensor = normalize_image(crop_tensor)
        
        # Add batch dimension
        crop_tensor = crop_tensor.unsqueeze(0)
        
        # Stage 2: ResNet inference (CPU)
        with torch.no_grad():
            outputs = resnet(crop_tensor)
            probs = torch.softmax(outputs, dim=1)
            confidence, local_class = torch.max(probs, dim=1)
        
        local_class = local_class.item()
        confidence = confidence.item()
        
        # Skip background (last class)
        bg_class = resnet.fc.out_features - 1
        if local_class == bg_class:
            continue
        
        # Map to actual sign name and global ID
        sign_name = CATEGORY_SIGNS[category][local_class]
        global_id = NAME_TO_ID[sign_name]
        
        detections.append({
            "bbox": [x1, y1, x2, y2],
            "class_id": global_id,
            "class_name": sign_name,
            "confidence": confidence
        })
    
    return detections

## CONFIG

In [77]:
#EVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#DEVICE = 'cpu'
print(f"Using device: {DEVICE}")

Using device: cpu


In [79]:
# Output directory for trained models
RESNET_SAVE_DIR = Path("../models/resnet_weights")
RESNET_SAVE_DIR.mkdir(parents=True, exist_ok=True)

## RESNET MODEL DEFINITION

In [82]:
def create_resnet_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    Create a ResNet18 model with custom FC layer for given number of classes.
    
    Args:
        num_classes: Number of classes (signs + 1 background)
        freeze_backbone: If True, freeze all layers except the final FC layer
    
    Returns:
        ResNet18 model ready for training
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Freeze backbone if requested
    if freeze_backbone:
        # Freeze all parameters initially
        for param in model.parameters():
            param.requires_grad = False
        
        # Unfreeze the final FC layer only
        # But FC layer doesn't exist yet, so we'll freeze after replacing
        # Actually: freeze all first, then replace FC (which will be trainable by default)
        # Let's do it cleanly:
        
        # Freeze all layers
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace the final fully connected layer
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    
    # FC layer is trainable by default (requires_grad = True)
    # If backbone was frozen, only FC will be trained
    
    return model

## TRAINING FUNCTION

In [85]:
def train_resnet(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 50,
    lr: float = 0.001,
    device: torch.device = DEVICE
) -> dict:
    """
    Train a ResNet model.
    
    Returns:
        Dictionary with training history
    """
    model = model.to(device)
    
    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        
        scheduler.step(avg_val_loss)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}")
        print(f"  Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), RESNET_SAVE_DIR / f"best_resnet_cat_{category_id}.pt")
            print(f"  -> Saved best model (acc: {val_acc:.4f})")
    
    return history

## TRAIN ALL 5 RESNET MODELS

In [104]:
def train_all_resnets(
    train_datasets: dict,   # CategoryDataset for training
    val_datasets: dict,     # CategoryDataset for validation
    batch_size: int = 32, 
    epochs: int = 50
):
    """
    Train 5 ResNet models using separate train and validation datasets.
    
    Args:
        train_datasets: Dictionary from build_resnet_dataset (TRAIN split)
        val_datasets: Dictionary from build_resnet_dataset (VAL split)
        batch_size: Training batch size
        epochs: Number of epochs per model
    """
    results = {}
    
    for category_id in range(5):
        print("\n" + "="*60)
        print(f"Training ResNet for Category {category_id}")
        print("="*60)
        
        # Get train and val samples for this category
        train_samples = train_datasets[category_id].samples
        val_samples = val_datasets[category_id].samples
        
        if len(train_samples) == 0:
            print(f" No training samples for category {category_id}, skipping...")
            continue
        
        # Find number of classes from training data
        max_class = -1
        for s in train_samples:
            if not s.is_background and s.sign_class > max_class:
                max_class = s.sign_class
        
        # Total classes = signs + 1 background
        num_classes = max_class + 2
        
        print(f"Category {category_id}:")
        print(f"  - Sign classes: {num_classes - 1}")
        print(f"  - Background class: {num_classes - 1} (last class)")
        print(f"  - Total classes: {num_classes}")
        print(f"  - Train samples: {len(train_samples)}")
        print(f"  - Val samples: {len(val_samples)}")
        
        # Create datasets
        train_dataset = ResNetDataset(train_samples, num_classes=num_classes, augment=True)
        val_dataset = ResNetDataset(val_samples, num_classes=num_classes, augment=False)
        
        # Create dataloaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Create model
        model = create_resnet_model(num_classes, freeze_backbone=True)
        
        # Train
        history = train_resnet(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs
        )
        
        results[category_id] = {
            'model': model,
            'history': history,
            'num_classes': num_classes
        }
        
        # Save final model
        torch.save(model.state_dict(), RESNET_SAVE_DIR / f"resnet_cat_{category_id}_final.pt")
        print(f" Saved final model for category {category_id}")
    
    return results

## SAVE CATEGORY MAPPING (for inference)

In [91]:
def save_category_mapping():
    """Save category to signs mapping as a JSON file."""
    import json
    
    mapping_data = {
        "category_to_signs": CATEGORY_SIGNS,
        "sign_name_to_id": NAME_TO_ID,
        "all_signs": ALL_SIGNS
    }
    
    with open(RESNET_SAVE_DIR / "category_mapping.json", 'w') as f:
        json.dump(mapping_data, f, indent=2)
    
    print(f" Saved category mapping to {RESNET_SAVE_DIR / 'category_mapping.json'}")

## VISUALIZE TRAINING HISTORY

In [94]:
def plot_training_history(results: dict):
    """Plot training loss and validation accuracy for all categories."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, (cat_id, data) in enumerate(results.items()):
        history = data['history']
        
        ax = axes[i]
        ax.plot(history['train_loss'], label='Train Loss')
        ax.plot(history['val_loss'], label='Val Loss')
        ax.set_title(f'Category {cat_id}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        
        ax2 = ax.twinx()
        ax2.plot(history['val_acc'], 'g--', label='Val Acc')
        ax2.set_ylabel('Accuracy')
        
        # Add best accuracy annotation
        best_acc = max(history['val_acc'])
        ax2.axhline(y=best_acc, color='gray', linestyle=':', alpha=0.5)
        ax2.text(epochs * 0.7, best_acc + 0.02, f'Best: {best_acc:.3f}', fontsize=9)
    
    # Hide unused subplot
    if len(results) < 5:
        axes[5].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(RESNET_SAVE_DIR / 'training_history.png', dpi=150)
    plt.show()
    print(f"Saved training history plot to {RESNET_SAVE_DIR / 'training_history.png'}")


## Create set 

In [97]:
# Quick version - just the mapping dictionary
sign_to_category = {
    0: 1,   # forb_ahead → category 1
    1: 1,   # forb_left → category 1
    2: 1,   # forb_overtake → category 1
    3: 1,   # forb_right → category 1
    4: 1,   # forb_speed_over_10 → category 1
    5: 1,   # forb_speed_over_100 → category 1
    6: 1,   # forb_speed_over_130 → category 1
    7: 1,   # forb_speed_over_20 → category 1
    8: 1,   # forb_speed_over_30 → category 1
    9: 1,   # forb_speed_over_40 → category 1
    10: 1,  # forb_speed_over_5 → category 1
    11: 1,  # forb_speed_over_50 → category 1
    12: 1,  # forb_speed_over_60 → category 1
    13: 1,  # forb_speed_over_70 → category 1
    14: 1,  # forb_speed_over_80 → category 1
    15: 1,  # forb_speed_over_90 → category 1
    16: 1,  # forb_stopping → category 1
    17: 1,  # forb_trucks → category 1
    18: 1,  # forb_u_turn → category 1
    19: 1,  # forb_weight_over_3.5t → category 1
    20: 1,  # forb_weight_over_7.5t → category 1
    21: 2,  # info_bus_station → category 2
    22: 2,  # info_crosswalk → category 2
    23: 2,  # info_highway → category 2
    24: 2,  # info_one_way_traffic → category 2
    25: 2,  # info_parking → category 2
    26: 2,  # info_taxi_parking → category 2
    27: 3,  # mand_bike_lane → category 3
    28: 3,  # mand_left → category 3
    29: 3,  # mand_left_right → category 3
    30: 3,  # mand_pass_left → category 3
    31: 3,  # mand_pass_left_right → category 3
    32: 3,  # mand_pass_right → category 3
    33: 3,  # mand_right → category 3
    34: 3,  # mand_roundabout → category 3
    35: 3,  # mand_straigh_left → category 3
    36: 3,  # mand_straight → category 3
    37: 3,  # mand_straight_right → category 3
    38: 4,  # prio_give_way → category 4
    39: 4,  # prio_priority_road → category 4
    40: 4,  # prio_stop → category 4
    41: 0,  # warn_children → category 0
    42: 0,  # warn_construction → category 0
    43: 0,  # warn_crosswalk → category 0
    44: 0,  # warn_cyclists → category 0
    45: 0,  # warn_domestic_animals → category 0
    46: 0,  # warn_other_dangers → category 0
    47: 0,  # warn_poor_road_surface → category 0
    48: 0,  # warn_roundabout → category 0
    49: 0,  # warn_slippery_road → category 0
    50: 0,  # warn_speed_bumper → category 0
    51: 0,  # warn_traffic_light → category 0
    52: 0,  # warn_tram → category 0
    53: 0,  # warn_two_way_traffic → category 0
    54: 0,  # warn_wild_animals → category 0
}

In [99]:
best_model_path = "../models/detect/cascade_model/weights/best.pt"
yolo = YOLO(str(best_model_path))

In [ ]:
# Train
train_boxes = count_boxes("../data/processed/train_balanced/labels/")
print(f"Train boxes: {train_boxes}")

# Test
test_boxes = count_boxes("../data/raw/Traffic Signs/valid/labels/")
print(f"Test boxes: {test_boxes}")

print(f"Total: {train_boxes + test_boxes}")

In [101]:
category_datasets_train = build_resnet_dataset(
    image_dir="../data/processed/train_balanced/images/",
    label_dir="../data/processed/train_balanced/labels/",
    yolo_model=yolo,
    sign_to_category=sign_to_category
)



Dataset statistics after building:
Category 0: 2073 positive, 888 background
Category 1: 3018 positive, 1293 background
Category 2: 1847 positive, 791 background
Category 3: 1660 positive, 711 background
Category 4: 894 positive, 383 background


In [102]:
category_datasets_val = build_resnet_dataset(
    image_dir="../data/raw/Traffic Signs/valid/images/",
    label_dir="../data/raw/Traffic Signs/valid/labels/",
    yolo_model=yolo,
    sign_to_category=sign_to_category
)



Dataset statistics after building:
Category 0: 182 positive, 78 background
Category 1: 352 positive, 150 background
Category 2: 194 positive, 83 background
Category 3: 232 positive, 99 background
Category 4: 333 positive, 142 background


## Teaching

In [ ]:
results = train_all_resnets(
        train_datasets=category_datasets_train,
        val_datasets=category_datasets_val,
        batch_size=16,
        epochs=50
    )
# Save category mapping for inference
save_category_mapping()
    
# Plot training history
plot_training_history(results)
    
print("\n" + "="*60)
print(" All ResNet models trained and saved!")
print(f"Models saved to: {RESNET_SAVE_DIR}")
print("="*60)
    
    # Summary
for cat_id, data in results.items():
    best_acc = max(data['history']['val_acc'])
    print(f"Category {cat_id}: Best val accuracy = {best_acc:.4f}")


Training ResNet for Category 0
Category 0:
  - Sign classes: 55
  - Background class: 55 (last class)
  - Total classes: 56
  - Train samples: 2961
  - Val samples: 260


Epoch 1/50 [Train]:   9%|█████▍                                                         | 8/93 [00:17<03:09,  2.23s/it]